# 模块十：定时质量监控 — Background §6.10 升级

## 学习目标

1. 理解从「module2 一次性 GE CLI」升级为「Checkpoint 抽象 + APScheduler 守护 + SQLite 历史 + 阈值告警」的动机与收益。
2. 跑通 `--run-once` 单次执行 5 系统 Checkpoint，写入 `data/quality_scores.db`。
3. 用 `query_history(days=30)` 查历史、用 `plot_trend_lines()` 渲染 5 系统折线图。
4. 模拟 score < 70 阈值告警：写一条 alert 到 `data/quality_alerts.json`（不入 DB）。

## 与 module2 的关系

| 维度 | module2 (Phase 1) | module10 (Phase 2 / 6.10) |
|---|---|---|
| 执行方式 | 手动跑 `scripts/run_great_expectations.py` | APScheduler 每日 08:30 + `--run-once` 手动 |
| 报告存储 | CLI 输出 + 单次 JSON | SQLite 时序表（`data/quality_scores.db`） |
| 告警 | 无 | `score < 70` 写 `data/quality_alerts.json` + `logging.warning` |
| 趋势 | 无 | matplotlib 5 系统折线图 `data/quality_trend.png` |
| 规则源 | `RULES` 字典 | **同一份** `RULES`（不重写） |

本 notebook 只依赖 **静态历史数据**（`data/historical/*.parquet`），不依赖 DataHub 服务，可在路径 A 下独立运行。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.quality_scheduler import run_all_systems, plot_trend_lines

results = run_all_systems()
print(f'✅ quality check done: {len(results)} tables checked')
for r in results:
    print(f'  {r.system}.{r.table:<8} score={r.score:>6.2f} grade={r.grade} passed={r.passed} failed={r.failed}')

In [ ]:
from scripts.quality_scheduler import query_history

df = query_history(days=30)
print(f'rows: {len(df)}, columns: {list(df.columns)}')
print()
print('Top 5 lowest scores:')
if not df.empty:
    print(df.sort_values('score').head(5).to_string(index=False))
else:
    print('(no history)')

In [ ]:
from IPython.display import Image, display
from pathlib import Path

png = plot_trend_lines(days=30)
if png is None:
    print('INFO quality_trend: no history yet, skipping')
else:
    abs_path = Path(png).resolve()
    size_kb = abs_path.stat().st_size / 1024
    print(f'PNG: {abs_path} ({size_kb:.1f} KB)')
    display(Image(filename=str(abs_path)))

In [ ]:
import json
import logging
from datetime import datetime
from pathlib import Path

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('simulate_alert')

ALERTS_PATH = Path('..') / 'data' / 'quality_alerts.json'
DB_PATH = Path('..') / 'data' / 'quality_scores.db'

simulated_score = 60.0
simulated_system = 'simulated_orphan_source'
log.warning(f'quality_alert system={simulated_system} score={simulated_score}')

alerts = []
if ALERTS_PATH.exists():
    alerts = json.loads(ALERTS_PATH.read_text(encoding='utf-8') or '[]')
new_entry = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'system': simulated_system,
    'score': simulated_score,
    'grade': 'D',
}
alerts.append(new_entry)
ALERTS_PATH.parent.mkdir(parents=True, exist_ok=True)
ALERTS_PATH.write_text(json.dumps(alerts, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'alerts.json: {ALERTS_PATH.resolve()}')
print(f'  new entry: {new_entry}')

import sqlite3
with sqlite3.connect(DB_PATH) as conn:
    polluted = list(conn.execute(
        'SELECT COUNT(*) FROM quality_scores WHERE score = ? AND system = ?',
        (simulated_score, simulated_system),
    ))[0][0]
print(f'quality_scores.db has score=60 rows for {simulated_system}: {polluted} (must be 0 — simulated data not in DB)')